# Finding an optimal $\hat \eta$ using a grid search

In [1]:
from flow_helpers import *

C:\Users\amirt\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
feathers = pathlib.Path(ds.__path__[0] + '/../../data/')

allpsrs = sorted(
    [ds.Pulsar.read_feather(psrfile) for psrfile in list(feathers.glob("*-[JB]*.feather"))],
    key=lambda psr: len(psr.toas), reverse = True
)

In [ ]:
npsr = 5
psrs = allpsrs[:npsr]

Tspan = ds.getspan(psrs)

In [5]:
rn_components = 30
gw_components = 14
powerlaw = ds.powerlaw

# here we fix the RN and WN, which should make the result exact, the WN is fixed through the psr.noisedict
rn_init_params = [{'log10_A': -12.0, 'gamma': 4.0}] * npsr #[{'log10_A': -11.0, 'gamma': 6.5}]*npsr # broad prior
tnequad = False
ecorr = False
fixed_wn = True

pslmodels = fouriermodel(psrs, rn_components, 
                                 rn_init_params= rn_init_params, ecorr = ecorr, tnequad = tnequad, 
                                 powerlaw = powerlaw, fixed_wn = fixed_wn)


In [6]:
psr_noisedicts = [psr.noisedict for psr in allpsrs]
L0s, ahat0_list, _ = compute_zero_quantities(pslmodels, psr_noisedicts)
L0s = jnp.stack(L0s)
ahat0_list = jnp.stack(ahat0_list)
Sigma_0_inv = jax.vmap(lambda L0: jnp.linalg.inv(L0 @ L0.T))(L0s)

In [7]:
b0s = jax.vmap(lambda L0, ahat0: jsp.linalg.cho_solve((L0, True), ahat0))(L0s, ahat0_list)

_, f, df = construct_freqs(psrs, num_frequencies=rn_components)

TtNT, _ = TtNT_mpsrs(Sigma_0_inv, params_list=[rn_init_params[0]] * npsr,
                                f=f, df=df, powerlaw=powerlaw)


In [8]:
b0s = jax.vmap(lambda L0, ahat0: jsp.linalg.cho_solve((L0, True), ahat0))(L0s, ahat0_list)

In [9]:
# Creating keys for the rn params + gw params
psrnames = [psr.name for psr in pslmodels]
rn_amp_keys, rn_gamma_keys = create_rn_keys(psrnames)

In [10]:
crn_gamma_key = "crn_gamma"
crn_log10A_key = "crn_log10_A"
crn_components = 14


# Marginalizing over CURN numerically

In [14]:
for psr_idx in range(npsr):
    psrs_subset = [psrs[psr_idx]]

    rn_amp_key_subset = [rn_amp_keys[psr_idx]]    
    rn_gamma_key_subset = [rn_gamma_keys[psr_idx]]  

    commongp = ds.makecommongp_fourier(psrs_subset, ds.powerlaw, rn_components,
                                        T=Tspan, name='red_noise')
    commongp_crn = ds.makecommongp_fourier(psrs_subset, ds.powerlaw,
                                            components=crn_components, T=Tspan, name='crn',
                                            common=[crn_log10A_key, crn_gamma_key],)

    getN_common = commongp.Phi.getN
    getN_crn = commongp_crn.Phi.getN

    phi_crn_args = (crn_components, rn_amp_key_subset, rn_gamma_key_subset,
                    crn_log10A_key, crn_gamma_key, getN_common, getN_crn)

    phi_crn_partial = jax.jit(lambda rho: phi_crn(rho, *phi_crn_args))

    log_posterior = make_marginalized_log_posterior(
        TtNT[psr_idx], b0s[psr_idx], phi_crn_partial, 
        rn_amp_key_subset, rn_gamma_key_subset,
    crn_log10A_key, crn_gamma_key, n_crn_grid=35)

    eta_0 = eta_MAP(log_posterior,n_grid = 20, steps = 5, zoom = 0.3)

    print(f"eta MAP for {psrs[psr_idx].name}: {jnp.round(jnp.array(eta_0), 2)}")

eta MAP for J1713+0747: [  1.64 -13.61]
eta MAP for J1909-3744: [  0.   -13.76]
eta MAP for J0636+5128: [  0.   -13.19]


KeyboardInterrupt: 

# Using a fixed CURN process

In [11]:
def make_log_posterior_fixedCRN(TtNT, b_0, phi_crn_partial,
                       rn_amp_keys, rn_gamma_keys,
                       crn_log10A_key, crn_gamma_key,
                       crn_params=[-14.0, 4.0]):

    crn_log10A, crn_gamma = crn_params[0], crn_params[1]

    def log_posterior(irn_log10A, irn_gamma):
        etas = {}
        for k in rn_amp_keys:
            etas[k] = irn_log10A
        for k in rn_gamma_keys:
            etas[k] = irn_gamma
        etas[crn_log10A_key] = crn_log10A
        etas[crn_gamma_key]  = crn_gamma

        phi_inv_diags, logdet_phi = phi_crn_partial(etas)
        phi_inv_diag = phi_inv_diags[0]

        Sigma_inv     = TtNT + jnp.diag(phi_inv_diag)
        L_inv         = jnp.linalg.cholesky(Sigma_inv)
        log_det_Sigma = -2.0 * jnp.sum(jnp.log(jnp.diag(L_inv)))
        mu            = jsp.linalg.cho_solve((L_inv, True), b_0)
        quad_b        = b_0 @ mu

        return 0.5*quad_b + 0.5*log_det_Sigma - 0.5*logdet_phi

    return log_posterior


for psr_idx in range(npsr):
    psrs_subset        = [psrs[psr_idx]]
    rn_amp_key_subset  = [rn_amp_keys[psr_idx]]
    rn_gamma_key_subset = [rn_gamma_keys[psr_idx]]

    commongp     = ds.makecommongp_fourier(psrs_subset, ds.powerlaw,
                       rn_components, T=Tspan, name='red_noise')
    commongp_crn = ds.makecommongp_fourier(psrs_subset, ds.powerlaw,
                       components=crn_components, T=Tspan, name='crn',
                       common=[crn_log10A_key, crn_gamma_key])

    phi_crn_args    = (crn_components, rn_amp_key_subset, rn_gamma_key_subset,
                       crn_log10A_key, crn_gamma_key,
                       commongp.Phi.getN, commongp_crn.Phi.getN)  # pass getN functions
    phi_crn_partial = jax.jit(lambda rho, args=phi_crn_args: phi_crn(rho, *args))

    log_posterior = make_log_posterior_fixedCRN(
        TtNT[psr_idx], b0s[psr_idx], phi_crn_partial,
        rn_amp_key_subset, rn_gamma_key_subset,
        crn_log10A_key, crn_gamma_key,
        crn_params=[-14.0, 4.0])

    eta_0 = eta_MAP(log_posterior, n_grid=10, steps=5, zoom=0.3)
    print(f"eta MAP for {psrs[psr_idx].name}: {jnp.round(jnp.array(eta_0), 2)}")

eta MAP for J1713+0747: [  0.51 -13.69]
eta MAP for J1909-3744: [  0.   -13.71]
eta MAP for J0636+5128: [  0.   -13.24]
eta MAP for J1012+5307: [  0.26 -12.57]
eta MAP for B1937+21: [  2.67 -13.21]
eta MAP for J1600-3053: [  0.62 -12.97]
eta MAP for J1643-1224: [  1.18 -12.24]
eta MAP for J0030+0451: [  0.   -19.94]
eta MAP for J1918-0642: [  1.38 -13.71]
eta MAP for J2145-0750: [  0.   -12.69]
